# Task 0 — The Library of Babel: Dataset Construction
**The Ghost in the Machine · PreCog NLP Task**

This notebook builds three **genre-matched, length-controlled** classes and two held-out
**probe sets**.

**Classes**
- Class 0 — Human: essayistic passages from two public-domain authors (Austen, Dickens)
- Class 1 — AI-neutral: LLM essays on the same topics, modern register
- Class 2 — AI-styled: LLM essays imitating the author, via few-shot exemplars

**Probe sets (predicted, never trained on)**: modern-human prose; my own SOP used to
*measure* genre leakage deliberately.

In [1]:
!pip install -q requests nltk google-genai gensim spacy scikit-learn groq
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import re, json, time, random
from pathlib import Path
import requests
random.seed(42)

In [ ]:
import os
MIN_WORDS, MAX_WORDS = 100, 275

PROJECT_ROOT = Path(r".")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR = PROJECT_ROOT / "data" / "dataset"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download human corpus from Project Gutenberg
We cache to disk so the run is reproducible and we never re-download. Both authors here are public domain.

In [4]:
GUTENBERG_BOOKS = {
    "austen":  [("pride_prejudice", "https://www.gutenberg.org/files/1342/1342-0.txt"),
                ("emma",            "https://www.gutenberg.org/files/158/158-0.txt")],
    "dickens": [("great_expectations", "https://www.gutenberg.org/files/1400/1400-0.txt"),
                ("tale_two_cities",    "https://www.gutenberg.org/files/98/98-0.txt"),
                ("oliver_twist", "https://www.gutenberg.org/files/730/730-0.txt")],
}

In [5]:
def download_book(name, url):
    cache = RAW_DIR / f"{name}.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8", errors="ignore")
    r = requests.get(url, timeout=60); r.raise_for_status()
    cache.write_text(r.text, encoding="utf-8", errors="ignore")
    return r.text

## 2. Clean Gutenberg boilerplate
The body sits between the `*** START ... ***` and `*** END ... ***` markers. If we don't strip
them, the human class is contaminated with modern legal/licensing English

In [6]:
def strip_gutenberg_boilerplate(text):
    start = re.search(r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text, re.I|re.S)
    if start: text = text[start.end():]
    end = re.search(r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text, re.I|re.S)
    if end: text = text[:end.start()]
    return text

In [7]:
def normalize_whitespace(text):
    text = text.replace("\r\n", "\n")
    paras = re.split(r"\n[ \t]*\n", text)
    return "\n\n".join(re.sub(r"\s+", " ", p).strip() for p in paras if p.strip())

## 3. Genre-matched, length-controlled extraction  ← the core decision
We KEEP essayistic/expository paragraphs (the narrator generalising about a theme) and DROP
narrative/dialogue (which is dense with quotation marks and character names). This is what makes
the human class the same *register* as the AI essays, so the detector can't cheat on genre or
proper nouns. We also enforce the [100, 200]-word window so length cannot leak.

In [8]:
ESSAYISTIC = re.compile(
    r"\b(it is|there is|there are|one must|we are|to be|such (a|an)|in general|always|never|"
    r"every|those who|that which|nothing (is|so)|truth|nature of|the world|society|mankind|human)\b", re.I)

In [9]:
def looks_narrative(p):
    if (p.count('"') + p.count("\u201c") + p.count("\u201d")) >= 2:   # dialogue
        return True
    toks = p.split()
    if not toks: return True
    interior_caps = sum(1 for t in toks[1:] if t[:1].isupper() and t[1:2].islower())
    return interior_caps / len(toks) > 0.08                          # character-name density

In [10]:
def extract_paragraphs(clean_text):
    out = []
    for p in clean_text.split("\n\n"):
        if not (MIN_WORDS <= len(p.split()) <= MAX_WORDS): continue
        if looks_narrative(p): continue
        if len(ESSAYISTIC.findall(p)) < 1: continue
        out.append(p)
    return out

In [11]:
def build_human_class(per_author=250):
    rows = []
    for author, books in GUTENBERG_BOOKS.items():
        pool = []
        for name, url in books:
            clean = normalize_whitespace(strip_gutenberg_boilerplate(download_book(name, url)))
            pool += [{"text": p, "author": author, "book": name} for p in extract_paragraphs(clean)]
        random.shuffle(pool); chosen = pool[:per_author]
        print(f"  {author}: {len(pool)} candidates -> kept {len(chosen)}")
        for r in chosen:
            r.update(label="human", class_id=0, word_count=len(r["text"].split()))
        rows += chosen
    return rows

In [12]:
human = build_human_class()
print("Total human:", len(human))
with open(OUT_DIR/"class1_human.jsonl", "w", encoding="utf-8") as f:
    for r in human: f.write(json.dumps(r, ensure_ascii=False)+"\n")

  austen: 216 candidates -> kept 216
  dickens: 208 candidates -> kept 208
Total human: 424


## 3b. Topic extraction (LDA) — derive topics, don't invent them
The task asks us to "identify 5-10 core topics." We DERIVE them from the human corpus with LDA so
they're defensible as *extracted*, not hand-picked. Each topic's top words become a human-readable
label, which then seeds generation — guaranteeing the AI classes are written on the *same* themes
the humans actually wrote about (key to the genre/topic match).

In [13]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

In [14]:
N_TOPICS = 10

In [15]:
EXTRA_STOP = {
    "mr","mrs","miss","sir","lady","little","good","time","day","man","men","woman","women",
    "way","came","come","come","went","got","knew","know","saw","look","looked","felt","feel",
    "did","like","long","away","poor","young","old","great","said","say","thing","things","make",
    "made","took","hand","eyes","face","head","room","night","home","house",
    "elizabeth","darcy","bennet","bingley","jane","wickham","collins","lydia","gardiner",
    "joe","pip","herbert","estella","havisham","pumblechook","magwitch",
    "emma","harriet","woodhouse","knightley","weston","fairfax","churchill", "hartfield", 
    "monseigneur", "hartfield", "going", "having", "soon", "better", "thought", "seen"
}
STOP = list(ENGLISH_STOP_WORDS.union(EXTRA_STOP))

In [16]:
def extract_topics(human_rows, n_topics=N_TOPICS, n_words=8):
    texts = [r["text"] for r in human_rows]
    vec = CountVectorizer(max_df=0.50, min_df=5, stop_words=STOP, ngram_range=(1, 2))
    X = vec.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=30)
    lda.fit(X)
    vocab = vec.get_feature_names_out()
    return [[vocab[i] for i in comp.argsort()[-n_words:][::-1]] for comp in lda.components_]

In [17]:
topic_words = extract_topics(human)
print("Raw LDA topics — INSPECT, then hand-write clean labels in TOPICS below:")
for k, w in enumerate(topic_words):
    print(f"  topic {k}: {', '.join(w)}")

Raw LDA topics — INSPECT, then hand-write clean labels in TOPICS below:
  topic 0: mind, love, far, oliver, think, spoke, sense, pleasure
  topic 1: door, sister, mother, father, heard, passed, child, wine
  topic 2: mind, half, able, glad, fagin, happiness, tried, heart
  topic 3: stood, followed, looking, place, small, distance, large, times
  topic 4: father, morning, passed, affection, highbury, days, present, aunt
  topic 5: oliver, light, door, window, looking, dark, turned, half
  topic 6: people, father, boy, large, frank, half, stood, left
  topic 7: mind, pleasing, elegance, feelings, idea, handsome, years, gentleman
  topic 8: daughter, life, leave, enscombe, kind, spirits, attention, perfectly
  topic 9: believed, general, wine, husband, known, sister, turned, farmer


## Unfortunate Circumstance
After multiple tries the topics extracted by LDA are not proper and I don't feel that i can use it for the final one, so takin the extracted topics I have build my own topics that work on the similar themes of what is extracted by LDA

In [18]:
# TOPICS = [f"the theme of {w[0]}, {w[1]} and {w[2]} in society" for w in topic_words]

In [19]:
TOPICS = [
    "family bonds and duty between parents and children",
    "marriage, courtship, and domestic life",
    "fear, dread, and moral terror",
    "affection, happiness, and disappointment",
    "childhood, innocence, and vulnerability",
    "social reputation and the judgement of others",
    "human nature, kindness, and moral character",
    "loss, separation, and leaving home",
    "wealth, inheritance, and social standing",
    "the influence of society on personal freedom",
]

In [20]:
print("\nGeneration topic labels:")
for t in TOPICS: print("  -", t)


Generation topic labels:
  - family bonds and duty between parents and children
  - marriage, courtship, and domestic life
  - fear, dread, and moral terror
  - affection, happiness, and disappointment
  - childhood, innocence, and vulnerability
  - social reputation and the judgement of others
  - human nature, kindness, and moral character
  - loss, separation, and leaving home
  - wealth, inheritance, and social standing
  - the influence of society on personal freedom


In [21]:
# Save the derived topics so the run is auditable.
with open(OUT_DIR/"extracted_topics.txt", "w", encoding="utf-8") as f:
    for k, (w, t) in enumerate(zip(topic_words, TOPICS)):
        f.write(f"{k}\t{t}\t[{', '.join(w)}]\n")

## Unfortunate Circumstance
Due to the rate limiting of gemini api key due to server stress, I have proceeded to shift from gemini to groq with "llama-3.3-70b-versatile" used as the model to generate the dataset

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "..."
STYLE_EXEMPLARS = {
    "austen": [
        "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.",
        "Vanity and pride are different things, though the words are often used synonymously. A person may be proud without being vain. Pride relates more to our opinion of ourselves, vanity to what we would have others think of us.",
        "Seldom, very seldom, does complete truth belong to any human disclosure; seldom can it happen that something is not a little disguised, or a little mistaken.",
    ],
    "dickens": [
        "It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity.",
        "It is a melancholy truth that even great men have their poor relations.",
        "There is a passion for hunting something deeply implanted in the human breast.",
    ],
}

In [23]:
def make_client():
    from groq import Groq
    import os
    return Groq(api_key=os.environ["GROQ_API_KEY"])

In [24]:
def generate_with_backoff(client, prompt, max_retries=8):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=1.0,
            )
            return resp.choices[0].message.content
        except Exception as e:
            if any(k in str(e) for k in ("429", "rate", "limit", "503", "502", "over capacity")):
                s = (2**attempt) + random.uniform(0, 1)
                print(f"    rate-limited; sleeping {s:.1f}s"); time.sleep(s)
            else:
                raise
    return None

In [25]:
def clean_generation(text):
    if not text: return None
    t = text.strip().strip('"').strip("\u201c\u201d").strip()
    t = re.sub(r"^\s*\d+[.)]\s*", "", t); t = re.sub(r"^#+\s*", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t if MIN_WORDS <= len(t.split()) <= MAX_WORDS else None

In [26]:
def neutral_prompt(topic):
    return (f"Write one self-contained paragraph of strictly {MIN_WORDS}-{MAX_WORDS} words (do not exceed 230 words) on "
            f"{topic}. Use clear, modern, expository prose. Do NOT use archaic or Victorian "
            f"language. Write ONLY the paragraph, no title or quotation marks.")

In [27]:
def styled_prompt(topic, author):
    ex = "\n".join(f"- {s}" for s in STYLE_EXEMPLARS[author])
    return (f"Sentences in the style to imitate:\n{ex}\n\nNow write one self-contained paragraph "
            f"of {MIN_WORDS}-{MAX_WORDS} (do not exceed 230 words) words discussing {topic}, closely imitating that author's "
            f"syntax, diction, and rhythm. Write ONLY the paragraph, no title or quotation marks.")

In [28]:
def generate_class(client, style, n_target=500):
    rows, attempts = [], 0
    authors = list(STYLE_EXEMPLARS)
    while len(rows) < n_target and attempts < n_target*4:
        topic = TOPICS[attempts % len(TOPICS)]
        author = authors[attempts % len(authors)] if style == "styled" else None
        prompt = styled_prompt(topic, author) if style == "styled" else neutral_prompt(topic)
        raw = generate_with_backoff(client, prompt)
        cleaned = clean_generation(raw)
        with open(OUT_DIR/"generation_log.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({"style": style, "topic": topic, "author": author,
                                "kept": cleaned is not None}, ensure_ascii=False)+"\n")
        if cleaned:
            rows.append({"text": cleaned, "author": f"{author}_styled" if author else "ai",
                         "topic": topic, "label": "ai_neutral" if style=="neutral" else "ai_styled",
                         "class_id": 1 if style=="neutral" else 2, "word_count": len(cleaned.split())})
            if len(rows) % 25 == 0: print(f"    {style}: {len(rows)}/{n_target}")
        attempts += 1; time.sleep(6)
    return rows

In [29]:
client = make_client()
neutral = generate_class(client, "neutral", 420)
styled  = generate_class(client, "styled", 420)
for name, data in [("class2_ai_neutral", neutral), ("class3_ai_styled", styled)]:
    with open(OUT_DIR/f"{name}.jsonl", "w", encoding="utf-8") as f:
        for r in data: f.write(json.dumps(r, ensure_ascii=False)+"\n")

    neutral: 25/420
    neutral: 50/420
    neutral: 75/420
    neutral: 100/420
    rate-limited; sleeping 1.9s
    neutral: 125/420
    neutral: 150/420
    neutral: 175/420
    rate-limited; sleeping 1.9s
    rate-limited; sleeping 2.5s
    neutral: 200/420
    neutral: 225/420
    neutral: 250/420
    neutral: 275/420
    neutral: 300/420
    neutral: 325/420
    neutral: 350/420
    rate-limited; sleeping 1.8s
    rate-limited; sleeping 2.6s
    rate-limited; sleeping 4.1s
    rate-limited; sleeping 8.1s
    rate-limited; sleeping 16.3s
    rate-limited; sleeping 32.9s
    rate-limited; sleeping 1.8s
    rate-limited; sleeping 2.9s
    rate-limited; sleeping 4.9s
    rate-limited; sleeping 8.2s
    rate-limited; sleeping 16.2s
    rate-limited; sleeping 32.1s
    rate-limited; sleeping 64.8s
    rate-limited; sleeping 128.9s
    rate-limited; sleeping 1.4s
    rate-limited; sleeping 1.6s
    rate-limited; sleeping 2.2s
    rate-limited; sleeping 4.9s
    rate-limited; sleeping 8.9

**Note**: This step took a lot of time due to groq's rate limiting(although not as bad as gemini's) so I've used the time to plan out the next steps.

In [30]:
client = make_client()
raw = generate_with_backoff(client, neutral_prompt(TOPICS[0]))
print(repr(raw)[:300], "| words:", len(raw.split()) if raw else "NONE")

    rate-limited; sleeping 1.7s
    rate-limited; sleeping 2.5s
    rate-limited; sleeping 4.8s
    rate-limited; sleeping 8.8s
    rate-limited; sleeping 16.3s
    rate-limited; sleeping 32.5s
"The relationship between parents and children is built on a foundation of mutual love, trust, and responsibility. As children grow and develop, they rely on their parents for guidance, support, and care, while parents are driven by a strong instinct to protect and provide for their offspring. This  | words: 173


## 5. Balance, freeze the split, sanity-check
Equal class sizes (no class-size leak). One stratified split written to disk and reused by
**every** tier, so cross-tier comparison is valid and there's no train/test contamination.

In [31]:
def load_jsonl(p): return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

In [32]:
def make_split(rows, ratios=(0.70, 0.15, 0.15), seed=42):
    by = {}
    for r in rows: by.setdefault(r["class_id"], []).append(r)
    rng = random.Random(seed); train=val=test=None; train, val, test = [], [], []
    for cid, items in by.items():
        rng.shuffle(items); n=len(items); a=int(n*ratios[0]); b=int(n*ratios[1])
        train += items[:a]; val += items[a:a+b]; test += items[a+b:]
    for name, part in [("train",train),("val",val),("test",test)]:
        with open(OUT_DIR/f"{name}.jsonl","w",encoding="utf-8") as f:
            for r in part: f.write(json.dumps(r, ensure_ascii=False)+"\n")
    print(f"  split -> train {len(train)} | val {len(val)} | test {len(test)}")

In [35]:
human   = load_jsonl(OUT_DIR/"class1_human.jsonl")
neutral = load_jsonl(OUT_DIR/"class2_ai_neutral.jsonl")
styled  = load_jsonl(OUT_DIR/"class3_ai_styled.jsonl")
n = min(len(human), len(neutral), len(styled))
all_rows = random.sample(human,n)+random.sample(neutral,n)+random.sample(styled,n)
with open(OUT_DIR/"dataset_full.jsonl","w",encoding="utf-8") as f:
    for r in all_rows: f.write(json.dumps(r, ensure_ascii=False)+"\n")
make_split(all_rows)

  split -> train 882 | val 189 | test 189


In [36]:
import statistics as st
for cid in (0,1,2):
    wcs=[r["word_count"] for r in all_rows if r["class_id"]==cid]
    print(f"class {cid}: n={len(wcs)} mean_words={st.mean(wcs):.1f} sd={st.pstdev(wcs):.1f}")

class 0: n=420 mean_words=153.9 sd=44.8
class 1: n=420 mean_words=166.7 sd=12.2
class 2: n=420 mean_words=145.4 sd=8.9


## 6. Building the probe sets
- `data/dataset/probe_modern_human.jsonl` — recent public-domain non-fiction / essays (NOT Victorian),
  100-200 words/paragraph, labelled human. Tests whether the detector generalises to modern human prose.
- `data/dataset/probe_sop.jsonl` — my own SOP/essay, split into paragraphs. The personal Turing test.

Both are **predicted but never trained on**. They are how we *measure* genre/era leakage
instead of discovering it by accident.

In [ ]:
url = "https://www.gutenberg.org/files/38280/38280-0.txt"
clean = normalize_whitespace(strip_gutenberg_boilerplate(download_book("modern_probe", url)))
paras = [p for p in clean.split("\n\n") if 100 <= len(p.split()) <= 200][:60]
with open("data/dataset/probe_modern_human.jsonl", "w", encoding="utf-8") as f:
    for i, p in enumerate(paras):
        f.write(json.dumps({"text": p, "label": "human", "class_id": 0,
                            "source": "modern_human", "word_count": len(p.split())}) + "\n")
print(f"wrote {len(paras)} modern-human paragraphs")

wrote 60 modern-human paragraphs


In [ ]:
import json
sop_text = """There are some labs you discover and think, “this is interesting.” And then there are a few rare
ones that make you feel, almost immediately, “this is where I belong.” PreCog is firmly the
second kind for me. Its work across responsible and safe AI, applied machine learning, NLP,
and social AI systems is exactly the kind of research space that excites me most today. What
draws me in is not just the technical depth of the lab, but the kind of questions it seems to care
about: not only whether AI systems are powerful, but whether they are understandable,
reliable, safe, and genuinely useful to people.

Over time, I have realized that this is the kind of researcher I want to become. I enjoy building
AI systems, but even more than that, I am fascinated by how they behave under pressure, how
they fail, how they can be measured honestly, and how they can be made trustworthy enough
for the real world. That is why a PhD at PreCog feels like such a natural next step for me. I do
not want to work only on models that perform well on paper; I want to work on AI systems that
can be evaluated rigorously, interpreted responsibly, and deployed with care.

A major reason for this excitement is Prof. Ponnurangam Kumaraguru. From everything I have
seen, PK sir feels like a larger-than-life figure in Indian AI research: a serious scholar, a builder
of institutions and communities, and someone who has managed to make responsible AI feel
both intellectually rigorous and socially urgent. His work at the intersection of responsible and
safe AI, applied machine learning, NLP, and socially meaningful systems strongly resonates
with the kind of research career I want to build. What I find especially inspiring is that his work
does not stay confined to papers alone; initiatives such as SARAL AI reflect a commitment to
democratizing research and making knowledge more accessible at scale.

My own journey into AI has been driven by a mix of curiosity, experimentation, and a very
strong desire to build things that matter. As an AI/ML engineer and applied researcher, I have
developed strong skills in Python, deep learning, experiment design, and model evaluation,
with a particular interest in responsible and governable AI systems. Across my internships and
projects, I have consistently found myself drawn not just to performance improvements, but to
questions of robustness, safety, explainability, and real-world usefulness.

At the University of Malaya, as an Exchange Research Student, I worked on a multimodal
athlete scouting assistant that combined vision and language components in a practical
screening workflow. The project reduced manual screening effort by around 40 percent, but
what mattered most to me was the process behind it: running controlled experiments over
thousands of annotated samples, comparing retrieval configurations, and understanding how
system behaviour changed under distribution shift. That experience taught me that practical AI
is not just about building a clever pipeline; it is about designing systems that continue to
behave sensibly when conditions become messy and realistic.

My internship at ISRO strengthened my appreciation for rigorous evaluation even further.
There, I worked on satellite image super-resolution and benchmarked our models over
45,000-plus image pairs across diverse land-cover types, carefully comparing them against
strong baselines. The technical gains were meaningful, but the bigger lesson for me was
learning how much discipline good research demands: controlled benchmarking, statistical
thinking, and honest assessment of where a model succeeds and where it breaks.

At IIT Madras, I worked on clinical NLP over large-scale structured health records, improving
entity extraction and note classification while also integrating uncertainty estimation, OOD
detection, explainability, and toxicity-aware filtering into the pipeline. That project had a deep
impact on how I think about AI. In healthcare, accuracy alone is not enough; a system must
also be auditable, interpretable, and safe for the people who depend on it. That way of thinking
now shapes how I approach almost every AI problem.

Beyond formal research roles, I have also spent a lot of time building and deploying applied AI
systems. Through my startup work and engineering roles, I have built multi-agent workflows,
RAG pipelines, and production AI systems designed for actual use rather than demos. These
experiences taught me how systems fail in practice, how users interact with them in
unexpected ways, and how important it is to combine engineering speed with research
discipline. I believe this builder’s instinct would be valuable at PreCog, where the goal is not
just to theorize about AI systems but to understand and improve them in the real world.

Another part of my journey that matters deeply to me is teaching. In my college, I have
conducted bootcamps for my juniors almost like an acting faculty member, helping them
understand generative AI and agentic AI development in a way that feels practical, exciting,
and accessible. I have found genuine joy in taking difficult technical ideas and making them
click for others. I will also soon be conducting corporate training, which I see as another
opportunity to sharpen this ability to communicate complex systems clearly and responsibly.
That teaching instinct is one more reason I feel aligned with a lab environment shaped by
someone like PK sir, who seems to combine research excellence with public-facing impact and
community building.

I am applying to PreCog because I want to grow into a researcher who can do both: ask deep
questions and build useful systems around them. I am especially interested in trustworthy AI
systems, responsible deployment of LLM-based and multimodal systems, interpretability,
evaluation under realistic failure settings, and safe applied ML for high-stakes domains. I want
to contribute to research that does not treat safety, transparency, or accountability as side
constraints, but as part of the core scientific problem.

If I join PreCog as a PhD student, I hope to bring energy, technical depth, seriousness, and a
strong willingness to work hard. I also hope to bring a sense of excitement. The kind that
comes from genuinely loving this field, from caring about what AI can become, and from
wanting to build systems that deserve people’s trust. PreCog feels like the kind of place where
that excitement would not just be welcomed, but sharpened into meaningful research.
More than anything, I want to do my PhD in an environment that feels alive: intellectually
ambitious, humane, collaborative, and deeply relevant to the world outside the lab. From
everything I have seen, PreCog feels exactly like that kind of place. It would be an honor to
learn under PK sir and contribute to a group that is thinking so seriously about the future of AI
and its impact on society."""
paras = [p.strip() for p in sop_text.split("\n\n") if len(p.split()) >= 40]
with open("data/dataset/probe_sop.jsonl", "w", encoding="utf-8") as f:
    for i, p in enumerate(paras):
        f.write(json.dumps({"text": p, "label": "human", "class_id": 0,
                            "source": "my_sop", "word_count": len(p.split())}) + "\n")
print(f"wrote {len(paras)} SOP paragraphs")

wrote 11 SOP paragraphs
